# GDELT (BigQuery) → extraction locale (raw)

Ce notebook exécute une requête BigQuery sur la table publique GDELT et sauvegarde le résultat dans `data/raw/`.

## Auth BigQuery (dans l'ordre)
- `GOOGLE_APPLICATION_CREDENTIALS` (chemin vers un JSON Service Account)
- `credentials/credentials.json` à la racine du repo
- sinon ADC (Application Default Credentials)

In [1]:
from __future__ import annotations
import logging
import sys
from pathlib import Path
sys.path.append(str(Path().resolve().parent))

import scripts.data_pipeline as dp

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s - %(message)s")
sys.path.append(str(Path().resolve().parent))

PROJECT_ROOT = Path().resolve().parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
import importlib
# Exécute le fichier data_pipeline.py et le recharge afin de s’assurer que les dernières modifications sont prises en compte dans la session en cours.
importlib.reload(dp)

<module 'scripts.data_pipeline' from 'E:\\junior\\Formations\\Hackathons\\Hackathon_iSHEEROXDatacamp\\scripts\\data_pipeline.py'>

In [2]:
events_query = """
SELECT *
FROM `gdelt-bq.gdeltv2.events` AS e
WHERE (
        (e.ActionGeo_CountryCode = 'BN' 
         AND LOWER(RTRIM(e.ActionGeo_FullName)) = 'benin' 
         AND e.ActionGeo_Type = 1)
     OR (e.Actor1Geo_CountryCode = 'BN' 
         AND LOWER(RTRIM(e.Actor1Geo_FullName)) = 'benin' 
         AND e.Actor1Geo_Type = 1)
     OR (e.Actor2Geo_CountryCode = 'BN' 
         AND LOWER(RTRIM(e.Actor2Geo_FullName)) = 'benin' 
         AND e.Actor2Geo_Type = 1)
      )
  AND e.SQLDATE BETWEEN 20250101 AND 20251231
""".strip()

client = dp.get_bq_client()
df = dp.extract_raw_data(client, events_query)

raw_dir = PROJECT_ROOT / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)
dp.load_raw_data(df, raw_dir / "gdelt_bn_2025.csv")
df.head()

2026-05-04 04:38:59,761 INFO scripts.data_pipeline - Using service account credentials from E:\junior\Formations\Hackathons\Hackathon_iSHEEROXDatacamp\credentials\credentials.json (project=model-caldron-495303-f7)
2026-05-04 04:38:59,763 INFO scripts.data_pipeline - Running query (511 chars)
2026-05-04 04:39:34,075 INFO scripts.data_pipeline - DataFrame saved to E:\junior\Formations\Hackathons\Hackathon_iSHEEROXDatacamp\data\raw\gdelt_bn_2025.csv


,GLOBALEVENTID,SQLDATE,MonthYear,Year,FractionDate,Actor1Code,Actor1Name,Actor1CountryCode,Actor1KnownGroupCode,Actor1EthnicCode,...,ActionGeo_Type,ActionGeo_FullName,ActionGeo_CountryCode,ActionGeo_ADM1Code,ActionGeo_ADM2Code,ActionGeo_Lat,ActionGeo_Long,ActionGeo_FeatureID,DATEADDED,SOURCEURL
0,1289290869,20250215,202502,2025,2025.1233,AFRCVL,AFRICA,AFR,NaN,NaN,...,1,Benin,BN,BN,NaN,9.5,2.25,BN,20260215143000,https://www.newsghana.com.gh/african-leaders-c...
1,1285645948,20250124,202501,2025,2025.0658,HLH,HOSPITAL,NaN,NaN,NaN,...,1,Benin,BN,BN,NaN,9.5,2.25,BN,20260124103000,https://punchng.com/senator-decries-poor-state...
2,1285645959,20250124,202501,2025,2025.0658,LEG,LAWMAKER,NaN,NaN,NaN,...,1,Benin,BN,BN,NaN,9.5,2.25,BN,20260124103000,https://punchng.com/senator-decries-poor-state...
3,1285645960,20250124,202501,2025,2025.0658,LEG,SENATE,NaN,NaN,NaN,...,1,Benin,BN,BN,NaN,9.5,2.25,BN,20260124103000,https://punchng.com/senator-decries-poor-state...
4,1283921640,20250114,202501,2025,2025.0384,GOVCVL,AMBASSADOR,NaN,NaN,NaN,...,1,Benin,BN,BN,NaN,9.5,2.25,BN,20260114201500,https://agenciabrasil.ebc.com.br/internacional...


In [ ]:
job = client.query(events_query)
print(job.state)
print(job.error_result)

In [4]:
gkg_query = """
SELECT 
    g.Date, 
    g.SourceCommonName, 
    g.Persons, 
    g.Organizations, 
    g.Locations, 
    g.Counts,
    g.V2Tone,
    g.TranslationInfo
FROM `gdelt-bq.gdeltv2.gkg` AS g
INNER JOIN `gdelt-bq.gdeltv2.events` AS e
  ON g.DocumentIdentifier = e.SOURCEURL
  AND DIV(g.Date, 1000000) = e.SQLDATE
WHERE e.SQLDATE BETWEEN 20250101 AND 20251231
  AND g.SourceCollectionIdentifier = 1
  AND (
        (e.ActionGeo_CountryCode = 'BN' AND LOWER(RTRIM(e.ActionGeo_FullName)) = 'benin' AND e.ActionGeo_Type = 1)
     OR (e.Actor1Geo_CountryCode = 'BN' AND LOWER(RTRIM(e.Actor1Geo_FullName)) = 'benin' AND e.Actor1Geo_Type = 1)
     OR (e.Actor2CountryCode = 'BN' AND LOWER(RTRIM(e.Actor2Geo_FullName)) = 'benin' AND e.Actor2Geo_Type = 1)
  )
""".strip()

## Recupération des données GKG
client = dp.get_bq_client()
raw_dir = PROJECT_ROOT / "data" / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

try:
  # Exécution de la requête
  job = client.query(gkg_query)
  job.result()  # force l'attente de la fin
  print("État du job :", job.state)

  # Extraction des données
  gkg_df = dp.extract_raw_data(client, gkg_query)

  # Sauvegarde du DataFrame dans un fichier CSV
  output_file = raw_dir / "gdelt_gkg_bn_2025.csv"
  dp.load_raw_data(gkg_df, output_file)

  # Vérification rapide
  print("Aperçu des données :")
  print(gkg_df.head())
except Exception as e:
  print("Erreur rencontrée :", e)
  if "job" in locals():
    print("Détails du job :", getattr(job, "error_result", None))


2026-05-04 14:17:59,557 INFO scripts.data_pipeline - Using service account credentials from E:\junior\Formations\Hackathons\Hackathon_iSHEEROXDatacamp\credentials\credentials.json (project=veriftel)
2026-05-04 14:18:43,675 INFO scripts.data_pipeline - Running query (743 chars)


État du job : DONE


2026-05-04 14:19:30,053 INFO scripts.data_pipeline - DataFrame saved to E:\junior\Formations\Hackathons\Hackathon_iSHEEROXDatacamp\data\raw\gdelt_gkg_bn_2025.csv


Aperçu des données :
             Date SourceCommonName  \
0  20251012190000       jagran.com   
1  20251012220000      thecable.ng   
2  20251012220000      thecable.ng   
3  20251012220000      thecable.ng   
4  20251012220000      thecable.ng   

                                             Persons  \
0  sivan a sun sharma;ashok mahato;gazipur a chau...   
1  frank azuekor;bayo onanuga;mike ozekhome;bola ...   
2  frank azuekor;bayo onanuga;mike ozekhome;bola ...   
3  frank azuekor;bayo onanuga;mike ozekhome;bola ...   
4  frank azuekor;bayo onanuga;mike ozekhome;bola ...   

                                       Organizations  \
0  visa the company a operations;visa;group the c...   
1  national open university;department of state s...   
2  national open university;department of state s...   
3  national open university;department of state s...   
4  national open university;department of state s...   

                                           Locations  \
0  4#Patna, Bihar, I